[<img src="imagens/colab-badge.png" style="width:16%; vertical-align:middle;">](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.fr/cap07/cap07_aluno.ipynb)
[<img src="imagens/github-badge.png" style="width:19%; vertical-align:middle;">](https://github.com/fzampirolli/pdi-vc)

# 7 Classification d'images et reconnaissance de formes

Dans le **chapitre 6**, la transition de la partie I vers la partie II a été présentée à travers deux applications qui exigeaient déjà des décisions automatisées : la reconnaissance de marques sur des feuilles de réponses (OMR) et la détection de défauts en inspection industrielle. Dans les deux cas, cependant, les décisions dépendaient de règles géométriques et de seuils définis manuellement, comme déterminer si un disque était suffisamment circulaire ou si une région était suffisamment sombre.

Ce chapitre formalise le problème plus général sous-jacent à ces applications : étant donné un ensemble d'exemples étiquetés, comment entraîner un système pour **classifier automatiquement** de nouvelles images ou régions d'intérêt ? Cette question est au cœur de la **reconnaissance de formes**, discipline qui fonde une grande partie des tâches modernes de vision par ordinateur, depuis la classification d'images jusqu'à la détection d'objets et la segmentation sémantique, explorées dans les chapitres suivants.

Seront étudiés les principaux **descripteurs classiques d'image** (couleur, texture et forme/gradient) ainsi que le classifieur **k-plus proches voisins** (*k-Nearest Neighbors* — k-NN), choisi pour sa simplicité conceptuelle et pour mettre en évidence, de manière directe, la relation entre l'espace des caractéristiques, les métriques de distance et les frontières de décision — des concepts qui demeurent centraux même dans les classifieurs fondés sur les réseaux de neurones profonds, étudiés dans le chapitre final de cette partie.

## 7.1 Objectifs du chapitre

À la fin de ce chapitre, l'étudiant devrait être capable de :

* **Comprendre le *pipeline* classique de reconnaissance de formes** : acquisition, prétraitement, extraction de descripteurs, classification et évaluation ;
* **Extraire et interpréter des descripteurs classiques** de couleur, de texture (*Local Binary Patterns* — LBP) et de forme/gradient (*Histogram of Oriented Gradients* — HOG) ;
* **Implémenter et entraîner un classifieur k-NN** pour des tâches de classification d'images ;
* **Évaluer les classifieurs** à l'aide de métriques telles que l'exactitude, la matrice de confusion, la précision et le rappel ;
* **Analyser l'effet du paramètre k** et de la dimensionnalité de l'espace des caractéristiques sur la performance du classifieur ;
* **Reconnaître les limites des descripteurs artisanaux** (*hand-crafted features*) et comprendre la motivation pour la transition, dans les chapitres suivants, vers des descripteurs appris automatiquement.

## 7.2 Configuration de l’environnement

Les exemples de ce chapitre utilisent des bibliothèques largement employées en traitement numérique des images, en vision par ordinateur et en apprentissage automatique. Le bloc ci-dessous installe les paquets nécessaires ; dans les environnements où ils sont déjà présents, l’exécution peut être ignorée.

In [1]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup()
from morph import mm

import importlib
import subprocess
import sys

def setup_cap07():
    """Installe les bibliothèques manquantes nécessaires pour ce chapitre
    (vision par ordinateur et apprentissage automatique)."""
    pacotes = {
        "cv2": "opencv-python",
        "skimage": "scikit-image",
        "numpy": "numpy",
        "sklearn": "scikit-learn",
        "matplotlib": "matplotlib",
        "pandas": "pandas",
        "seaborn": "seaborn",
        "tabulate": "tabulate",
        "kaleido": "kaleido",
    }
    for modulo, pacote in pacotes.items():
        if importlib.util.find_spec(modulo) is None:
            resultado = subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", pacote]
            )
            if resultado.returncode != 0:
                print(f"[AVERTISSEMENT] Échec de l'installation de {pacote} (nécessaire pour le module {modulo}).")


setup_cap07()

# ==========================================================
# Bibliothèques
# ==========================================================

# Calcul scientifique
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import seaborn as sns

# Vision par ordinateur
import cv2
from skimage import data as skdata
from skimage.feature import hog, local_binary_pattern

# Apprentissage automatique
from sklearn.datasets import load_digits, make_classification
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

✅ Environnement prêt. Morph : 1.1.9 | OpenCV : 5.0.0


## 7.3 Un Problème Concret : Classification des Fruits

Avant de présenter les fondements théoriques, considérons le problème suivant,
qui servira d'exemple tout au long de ce chapitre pour illustrer les
principaux concepts de la reconnaissance de formes.

**Le scénario :** Une ferme automatisée utilise un système de vision
par ordinateur pour séparer les pommes, les bananes et les oranges sur des
lignes d'emballage.

**Le défi :** Les fruits arrivent sur le tapis roulant dans différentes positions et
orientations, sous des conditions d'éclairage qui peuvent varier. De plus,
les feuilles, les ombres et les petites occlusions peuvent rendre leur identification difficile.
Comment développer un système capable de les classer correctement ?

**Une approche possible :**

1. Extraire des descripteurs qui représentent des caractéristiques pertinentes des
   fruits :
   - **Couleur :** distribution prédominante des couleurs ;
   - **Texture :** différences à la surface de la peau ;
   - **Forme :** caractéristiques géométriques du contour.

2. Entraîner un classificateur en utilisant des exemples préalablement étiquetés.

3. Utiliser le modèle entraîné pour classer automatiquement de nouveaux
   fruits.

La [Figure 7.1](#fig-07-frutas-motivacao) illustre, de manière conceptuelle, comment différentes
fruits peuvent être représentés dans un espace de caractéristiques
tridimensionnel.

> ### 💡 Réfléchissez avant de continuer
>
> Si chaque fruit était représenté uniquement par les valeurs d'intensité de
> ses pixels, serait-il possible de les distinguer de manière fiable ? Quels types
> d'informations pourrait-on extraire de l'image pour faciliter cette tâche ?

In [2]:
np.random.seed(42)

centros = {
    "Maçã":    [0.8, 0.2, 0.9],
    "Banana":  [0.3, 0.1, 0.2],
    "Laranja": [0.9, 0.8, 0.8],
}

fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(projection="3d")

for fruta, centro, cor, marcador in zip(
    centros,
    centros.values(),
    ["red", "gold", "orange"],
    ["o", "s", "^"],
):
    X = np.clip(np.random.normal(centro, 0.08, (70, 3)), 0, 1)
    ax.scatter(X[:, 0], X[:, 1], X[:, 2],
               c=cor, marker=marcador, s=35,
               alpha=0.7, label=fruta)

ax.set(
    xlim=(0,1), ylim=(0,1), zlim=(0,1),
    xlabel="Intensidade de cor",
    ylabel="Textura",
    zlabel="Forma",
    title="Espaço de Características"
)
ax.view_init(elev=25, azim=-60)
ax.legend(title="Frutas")
ax.zaxis.labelpad = 0.01

plt.tight_layout()
plt.show()

<Figure size 1500x1500 with 1 Axes>

**Figure 7.1:** Exemple motivationnel : différents fruits formant des regroupements distincts dans un espace de caractéristiques.


## 7.4 🗺️ Aperçu du Chapitre : Le *Pipeline* Classique de Classification d'Images

Avant de poursuivre, il est utile de présenter une vue d'ensemble intégrée de ce qui
sera étudié. La [Figure 7.2](#fig-07-infografo) illustre le flux général d'un système
classique de classification d'images, depuis l'image d'entrée jusqu'à
l'étape d'attribution du label final. Dans les sections suivantes, chacune des
étapes de ce processus sera étudiée en détail.

<figure id="fig-07-infografo" style="text-align:center; margin:1em 0;">
  <img src="imagens/fig-07-infografo.png" alt="" style="max-width:60%; display:block; margin:auto;" />
  <figcaption><strong>Figure 7.2:</strong> Aperçu du *pipeline* classique de classification d'images : extraction de descripteurs (LBP, HOG), formation de l'espace des caractéristiques et classification via k-NN. Ne s'applique pas aux modèles d'apprentissage profond (CNN, YOLO), qui apprennent des *features* de bout en bout directement à partir des pixels. **Source :** élaboré avec l'aide de Gemini Notebook ({GOOGLE}, 2025).</figcaption>
</figure>

> ### 📝 Portée de ce chapitre
>
> Dans ce chapitre, l'accent est mis exclusivement sur le classificateur k-NN,
> en raison de sa simplicité pédagogique et de sa capacité à illustrer de manière intuitive le
> concept d'espace des caractéristiques. D'autres classificateurs
> traditionnels largement utilisés en reconnaissance de formes — tels que
> les arbres de décision, les règles de classification et les machines à vecteurs de
> support (SVM) — sont discutés en profondeur dans
> Quilici-gonzalez (2014), notamment dans sa 2e édition, actuellement
> en production (QUILICI-GONZALEZ, 2026).

## 7.5 Fondements de la reconnaissance de formes

Un système de **reconnaissance de formes** a pour objectif d'attribuer une
catégorie (étiquette) à une observation — une image entière, une région
d'intérêt ou un signal — sur la base d'exemples préalablement étiquetés. De
manière générale, ce processus est organisé selon les étapes suivantes :

1. **Acquisition :** obtention de l'image ou du signal à classer ;
2. **Prétraitement :** normalisation, suppression du bruit, correction
   géométrique ou d'éclairage — étapes déjà étudiées dans les chapitres
   précédents ;
3. **Extraction de descripteurs (*features*) :** transformation de l'image en
   un **vecteur de caractéristiques** de dimension fixe, qui représente les
   propriétés pertinentes pour la tâche de classification ;
4. **Classification :** application d'un modèle qui associe le vecteur de
   caractéristiques à une classe ;
5. **Évaluation :** analyse des performances du modèle sur un ensemble de
   données indépendant de celui utilisé pour l'entraînement.

L'ensemble de tous les vecteurs de caractéristiques possibles constitue
l'**espace de caractéristiques** (*feature space*). Un bon descripteur produit
des représentations qui rapprochent, dans cet espace, les observations d'une
même classe et éloignent les observations de classes distinctes. Cette
propriété favorise les méthodes de classification basées sur la proximité,
comme le **k-NN**, et bénéficie également à divers autres classifieurs.

La [Figure 7.1](#fig-07-frutas-motivacao) illustre ce concept de manière schématique :
chaque fruit est représenté par un point dans un espace de caractéristiques à
trois dimensions (couleur, texture et forme). Bien que cet espace ne soit
qu'une simplification didactique, il montre comment les échantillons d'une
même classe tendent à former des regroupements, tandis que des classes
différentes occupent des régions distinctes, facilitant ainsi la tâche de
classification.

## 7.6 Extraction de descripteurs classiques

Avant la popularisation des réseaux de neurones profonds, les descripteurs d'image étaient, pour la plupart, conçus manuellement par des experts (*hand-crafted features*), sur la base de propriétés statistiques ou géométriques connues. Trois familles classiques sont particulièrement pertinentes :

* **Descripteurs de couleur :** histogrammes d'intensité ou de teinte, qui capturent la distribution des valeurs de couleur d'une région, déjà introduits au **Chapitre 3** via la fonction `mm.hist` ;
* **Descripteurs de texture :** capturent des motifs locaux de répétition, de rugosité ou d'orientation, comme le *Local Binary Patterns* (LBP), étudié ci-après ;
* **Descripteurs de forme/gradient :** décrivent la distribution des contours et des orientations du gradient, comme le *Histogram of Oriented Gradients* (HOG), largement utilisé dans la détection de personnes et d'autres objets.

Pour comparer l'information capturée par chaque approche, la [Figure 7.3](#fig-07-visualizacao-descritores) montre comment différentes techniques « voient » la même image.

In [3]:
# Charger une image d'exemple
imagem = skdata.camera()

# Appliquer les descripteurs
lbp_img = local_binary_pattern(
    imagem, 
    P=8, 
    R=1, 
    method="uniform"
    )

# Convertir le LBP en RGB uniquement pour faciliter la visualisation
lbp_norm = (lbp_img - lbp_img.min()) / (lbp_img.max() - lbp_img.min() + 1e-8)
lbp_rgb = (cm.nipy_spectral(lbp_norm)[..., :3] * 255).astype("uint8")

hog_features, hog_img = hog(
    imagem,
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    visualize=True,
)

# Affichage standardisé
mm.show(
    [imagem, lbp_rgb, hog_img],
    titles=[
        "Image originale\n(comme l'humain la voit)",
        "LBP : Texture\n(chaque couleur = un code LBP)",
        "HOG : Gradients et contours\n(zones claires = intensité plus élevée)",
    ],
    cols=3,
    figsize=(12, 4),
)

print("Observez comment chaque descripteur met en évidence des propriétés différentes :")
print("• LBP : met en évidence les motifs locaux de texture.")
print("• HOG : met en évidence les contours et les orientations des bords.")
print("• Image originale : contient uniquement les valeurs d'intensité.")

<Figure size 1800x600 with 3 Axes>

**Figure 7.3:** Comparaison visuelle de différents descripteurs appliqués à la même image. Chaque descripteur révèle des aspects distincts de la scène.


Observez comment chaque descripteur met en évidence des propriétés différentes :
• LBP : met en évidence les motifs locaux de texture.
• HOG : met en évidence les contours et les orientations des bords.
• Image originale : contient uniquement les valeurs d'intensité.


### 7.6.1 *Local Binary Patterns* (LBP)

Le LBP est un descripteur de texture qui code, pour chaque pixel central
$g_c$, la relation entre son intensité et celle des $P$ voisins disposés
dans un voisinage circulaire de rayon $R$ :

$$
\mathrm{LBP}_{P,R}(x_c, y_c) = \sum_{p=0}^{P-1} s(g_p - g_c)\, 2^p,
\qquad
s(z) =
\begin{cases}
1, & z \geq 0 \\
0, & z < 0
\end{cases}
$$

où :

- $(x_c, y_c)$ sont les coordonnées du pixel central ;
- $g_c$ est l'intensité du pixel central ;
- $g_p$ est l'intensité du $p$-ième pixel voisin ;
- $P$ est le nombre de voisins considérés ;
- $R$ est le rayon du voisinage circulaire ;
- $p$ est l'indice du voisin, avec $p = 0, 1, \ldots, P-1$ ;
- $s(z)$ est la fonction de seuillage définie dans l'équation, où $z = g_p - g_c$ ;
  elle prend la valeur 1 lorsque $z \geq 0$ et 0 lorsque $z < 0$ ;
- $2^p$ correspond au poids binaire associé au $p$-ième voisin.

Le code LBP obtenu décrit le motif local de contraste autour du pixel. L'histogramme de ces codes forme un vecteur de caractéristiques compact pour représenter la texture de l'image ([Figure 7.4](#fig-07-vetor-lbp)). Dans ce chapitre, on utilise la variante **uniforme**, qui regroupe les motifs non uniformes dans une seule catégorie, réduisant la dimensionnalité et augmentant la robustesse du descripteur.

In [4]:
plt.figure(figsize=(6, 4))

plt.hist(
    lbp_img.ravel(),
    bins=np.arange(-0.5, lbp_img.max() + 1.5, 1),
    density=True,
    edgecolor="black",
)

plt.title("Histograma dos códigos LBP")
plt.xlabel("Código LBP")
plt.ylabel("Frequência relativa")
plt.xticks(range(int(lbp_img.max()) + 1))
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

<Figure size 1800x1200 with 1 Axes>

**Figure 7.4:** Histogramme des codes LBP de l


> ### 💡 Fonction `local_binary_pattern`
>
> L’implémentation utilisée dans ce chapitre est fournie par la bibliothèque
> `scikit-image` :
>
> ```python
> local_binary_pattern(
>     imagem,
>     P=8,
>     R=1,
>     method="uniform"
> )
> ```
>
> où :
>
> * `image` : image en niveaux de gris ;
> * `P` : nombre de voisins également espacés dans le voisinage circulaire ;
> * `R` : rayon du voisinage, en pixels ;
> * `method` : stratégie de codage. Dans ce chapitre, on utilise la valeur
>   `"uniform"`.
>
> L’équation présentée précédemment décrit le **LBP original**. Dans
> l’implémentation retenue dans ce chapitre, l’option `method="uniform"`
> calcule initialement ce code, puis remappe les motifs non
> uniformes vers une seule catégorie, réduisant la dimensionnalité du
> descripteur et le rendant plus robuste aux petites variations locales.

La [Figure 7.3](#fig-07-visualizacao-descritores) présente la représentation visuelle du LBP, tandis que la [Figure 7.4](#fig-07-vetor-lbp) montre l’histogramme des codes LBP utilisé comme vecteur de caractéristiques.

Le **Projet Pratique 2** (section **Comparaison de Descripteurs pour la Classification de Textures**) emploie le LBP dans la classification de différents types de texture synthétique.

### 7.6.2 *Histogram of Oriented Gradients* (HOG)

Le HOG est un descripteur qui représente la forme d'un objet à travers la
distribution des orientations du gradient local. Comme pour l'opérateur
de Canny (**Chapitre 6**), on calcule initialement le gradient :

$$
|\nabla f(x,y)| =
\sqrt{\left(\frac{\partial f}{\partial x}\right)^2 +
      \left(\frac{\partial f}{\partial y}\right)^2},
\qquad
\theta(x,y) =
\operatorname{atan2}\!\left(
\frac{\partial f}{\partial y},
\frac{\partial f}{\partial x}
\right).
$$

où :

- $f(x,y)$ est l'intensité de l'image au pixel $(x,y)$ ;
- $\frac{\partial f}{\partial x}$ et $\frac{\partial f}{\partial y}$ sont,
  respectivement, les dérivées partielles de l'image dans les directions
  horizontale et verticale ;
- $|\nabla f(x,y)|$ est la magnitude du vecteur gradient au pixel $(x,y)$,
  indiquant l'intensité de la variation locale de l'image ;
- $\theta(x,y)$ est l'orientation du vecteur gradient au pixel $(x,y)$,
  calculée par la fonction $\operatorname{atan2}$, dont le résultat appartient à
  l'intervalle $(-\pi,\pi]$.

Bien que $\theta(x,y)$, tel que calculé par la fonction $\operatorname{atan2}$, appartienne à l'intervalle $(-\pi,\pi]$, l'implémentation standard du HOG utilise le **gradient non signé** (*unsigned*) : les orientations opposées (par exemple, $0$ et $\pi$) sont traitées comme équivalentes, et les angles sont mappés sur l'intervalle $[0,\pi)$ avant la construction de l'histogramme. Ce choix rend le descripteur invariant à la direction du contraste (par exemple, un bord clair-sombre et un bord sombre-clair produisent la même orientation).

L'image est ensuite divisée en **cellules** (*cells*). Pour chaque cellule,
on construit un histogramme des orientations du gradient, pondéré par la
magnitude correspondante. La concaténation des histogrammes de toutes les
cellules forme le vecteur de caractéristiques HOG, qui représente la
distribution spatiale des orientations du gradient et capture des informations
sur la forme et les contours de l'objet ([Figure 7.5](#fig-07-vetor-hog)).

In [5]:
n = 100

plt.figure(figsize=(8, 3))
plt.bar(
    range(n),
    hog_features[:n],
    width=0.9
)

plt.title("Primeiros componentes do vetor HOG")
plt.xlabel(f"Índice do componente (0–{n-1}, de um total de {hog_features.shape[0]})")
plt.ylabel("Valor normalizado")
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

<Figure size 2400x900 with 1 Axes>

**Figure 7.5:** Premiers 100 composants du vecteur de caractéristiques HOG.


> ### 💡 Fonction `hog`
>
> L'extraction du descripteur HOG est réalisée par la fonction :
>
> ```python
> hog(
>     image,
>     orientations=9,
>     pixels_per_cell=(8, 8),
>     cells_per_block=(2, 2),
>     visualize=True,
> )
> ```
>
> Les principaux paramètres sont :
>
> * `image` : image d'entrée ;
> * `orientations` : nombre de divisions angulaires de l'histogramme des orientations dans chaque cellule ;
> * `pixels_per_cell` : taille, en pixels, de chaque cellule où l'histogramme est calculé ;
> * `cells_per_block` : nombre de cellules utilisées pour la normalisation du descripteur ;
> * `visualize` : lorsque `True`, retourne également une image illustrant les gradients utilisés par le HOG.
>
> L'équation présentée précédemment décrit le calcul de la magnitude et de
> l'orientation du gradient, qui constituent la base du descripteur HOG. Dans
> l'implémentation adoptée dans ce chapitre, la fonction `hog()` utilise ces
> informations pour construire des histogrammes d'orientations dans chaque cellule de
> l'image, puis effectue la normalisation par blocs (`cells_per_block`),
> réduisant la sensibilité du descripteur aux variations d'éclairage et de contraste.


La [Figure 7.3](#fig-07-visualizacao-descritores) présente la représentation visuelle du HOG, tandis que la [Figure 7.5](#fig-07-vetor-hog) illustre les premiers composants du vecteur de caractéristiques extrait de l'image.

Le **Projet Pratique 1** (section **Classification de Chiffres Manuscrits avec k-NN**)
compare les performances des descripteurs HOG avec l'utilisation directe des intensités
des pixels comme vecteur de caractéristiques.

### 7.6.3 L'Impact de l'Échelle et la Normalisation des Caractéristiques

Le classificateur $k$-NN prend ses décisions en se basant sur la distance entre les vecteurs de caractéristiques. Par conséquent, l'échelle de chaque caractéristique influence directement le résultat de la classification. Si une variable présente des valeurs beaucoup plus grandes que les autres (par exemple, une intensité de couleur variant de $0$ à $255$, alors qu'un indice de circularité varie de $0$ à $1$), elle tend à dominer le calcul de la distance, réduisant ainsi l'influence des autres descripteurs.

Pour éviter ce problème, on applique une étape de **normalisation des caractéristiques**, généralement au moyen de la standardisation (*Z-score standardization*). Dans cette procédure, chaque caractéristique a désormais une moyenne égale à zéro et un écart-type égal à un, rendant comparables des grandeurs mesurées à l'origine sur des échelles différentes.

La standardisation est réalisée par la transformation

$$
z = \frac{x - \mu}{\sigma},
$$

où :

- $x$ est la valeur originale de la caractéristique ;
- $\mu$ est la moyenne de cette caractéristique calculée sur l'ensemble d'entraînement ;
- $\sigma$ est l'écart-type de la caractéristique ;
- $z$ est la valeur standardisée.

Après cette transformation, toutes les caractéristiques possèdent une moyenne égale à zéro et un écart-type égal à un, ce qui leur permet de contribuer de manière équilibrée au calcul des distances.

> ### 💡 Classe `StandardScaler`
>
> La standardisation utilisée dans ce chapitre est réalisée par la classe `StandardScaler`, de la bibliothèque `scikit-learn` :
>
> ```python
> from sklearn.preprocessing import StandardScaler
>
> scaler = StandardScaler()
> X_norm = scaler.fit_transform(X)
> ```
>
> où :
>
> * `StandardScaler()` : crée l'objet responsable de la standardisation ;
> * `fit_transform(X)` : calcule la moyenne et l'écart-type de chaque caractéristique de l'ensemble `X` et retourne la matrice standardisée.
>
> En pratique, la méthode `fit_transform()` exécute deux étapes : d'abord (`fit`), elle estime la moyenne ($\mu$) et l'écart-type ($\sigma$) de chaque caractéristique ; ensuite (`transform`), elle applique la transformation de standardisation présentée précédemment à toutes les valeurs de la matrice d'entrée.

La [Figure 7.6](#fig-07-normalizacao-features) montre l'effet de la normalisation.
Visuellement, la distribution des points reste la même ; ce qui change, c'est l'échelle des axes. Sans normalisation, la caractéristique de plus grande magnitude domine le calcul des distances entre les échantillons. Après la standardisation, toutes les caractéristiques contribuent de manière équilibrée au calcul des distances utilisées par le classificateur $k$-NN.

In [6]:
# Dados sintéticos com escalas muito diferentes
np.random.seed(42)
X_demo = np.random.randn(20, 2) * [100, 1]
y_demo = np.array([0] * 10 + [1] * 10)

print("Effet de la normalisation :")
print("  Caractéristique 1 : échelle ≈ 100")
print("  Caractéristique 2 : échelle ≈ 1")
print("\nSans normalisation, la première caractéristique domine le calcul des distances.")
print("Avec normalisation, les deux contribuent de manière équilibrée.")
print("\nLa normalisation est essentielle lorsque les caractéristiques possèdent des échelles différentes.")
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Sem normalização
axes[0].scatter(
    X_demo[y_demo == 0, 0], X_demo[y_demo == 0, 1],
    c="blue", label="Classe 0"
)
axes[0].scatter(
    X_demo[y_demo == 1, 0], X_demo[y_demo == 1, 1],
    c="red", label="Classe 1"
)
axes[0].set_title("Sem Normalização\n(escalas diferentes)")
axes[0].set_xlabel("Característica 1 (escala 100)")
axes[0].set_ylabel("Característica 2 (escala 1)")
axes[0].legend()

# Com normalização
scaler = StandardScaler()
X_norm = scaler.fit_transform(X_demo)

axes[1].scatter(
    X_norm[y_demo == 0, 0], X_norm[y_demo == 0, 1],
    c="blue", label="Classe 0"
)
axes[1].scatter(
    X_norm[y_demo == 1, 0], X_norm[y_demo == 1, 1],
    c="red", label="Classe 1"
)
axes[1].set_title("Com Normalização\n(características balanceadas)")
axes[1].set_xlabel("Característica 1")
axes[1].set_ylabel("Característica 2")
axes[1].legend()

plt.tight_layout()
plt.show()

Effet de la normalisation :
  Caractéristique 1 : échelle ≈ 100
  Caractéristique 2 : échelle ≈ 1

Sans normalisation, la première caractéristique domine le calcul des distances.
Avec normalisation, les deux contribuent de manière équilibrée.

La normalisation est essentielle lorsque les caractéristiques possèdent des échelles différentes.


<Figure size 3000x1200 with 2 Axes>

**Figure 7.6:** Importância da normalização das características para o classificador k-NN.


## 7.7 📌 Carte conceptuelle

Jusqu’ici, ont été présentés les descripteurs classiques (LBP, HOG, pixels
bruts) et la manière dont ils organisent les échantillons dans un espace de
caractéristiques. La [Figure 7.7](#fig-07-mapa-conceitual) synthétise ce parcours et
situe ces étapes dans le flux général d’un système classique de
classification d’images, en indiquant également les étapes suivantes —
classification (k-NN) et évaluation des
résultats — qui seront formalisées dans les sections suivantes.

<figure id="fig-07-mapa-conceitual" style="text-align:center; margin:1em 0;">
  <img src="imagens/fig-07-mapa-conceitual.png" alt="" style="max-width:60%; display:block; margin:auto;" />
  <figcaption><strong>Figure 7.7:</strong> Carte conceptuelle du processus de classification d’images utilisant des descripteurs classiques (LBP, HOG, pixels bruts) et un classifieur traditionnel (k-NN). Ne s’applique pas aux modèles d’apprentissage profond (CNN, YOLO), qui apprennent des *features end-to-end* directement à partir des pixels.</figcaption>
</figure>

## 7.8 Descripteurs en pratique

Après avoir pris connaissance des principaux descripteurs classiques, il est naturel de se demander comment ils influencent les performances d'un classificateur dans des situations proches de celles rencontrées en pratique.

Dans cette section, on compare l'utilisation de trois représentations distinctes des mêmes images : les intensités des pixels, les descripteurs LBP et les descripteurs HOG. Pour rendre l'expérience plus réaliste, on ajoute un bruit synthétique aux données, à des intensités différentes pour chaque descripteur — une forme simplifiée de simulation du fait qu'en pratique, différentes représentations tolèrent de manière inégale les imperfections de la capture (bruit du capteur, petites variations de position, etc.).

La [Figure 7.8](#fig-07-matrizes-confusao) présente les matrices de confusion obtenues pour chaque descripteur, permettant d'identifier dans quelles classes se produisent les principales erreurs de classification.
L'interprétation de ces matrices a été introduite au **Chapitre 1**, lorsque les concepts de **Vrai Positif (VP)**, **Faux Positif (FP)**, **Vrai Négatif (VN)** et **Faux Négatif (FN)** ont été présentés.
Ces concepts ont été explorés dans les **EPs 01_02** (métriques de classification) et **01_03** (*mean Average Precision* – mAP), disponibles sur :

- <https://fzampirolli.github.io/pdi-vc/eps/py.pt/EP01_02.html>
- <https://fzampirolli.github.io/pdi-vc/eps/py.pt/EP01_03.html>

Dans ce chapitre, les matrices de confusion sont employées pour analyser comment différents descripteurs influencent les performances du classificateur.

Ensuite, la [Figure 7.9](#fig-07-comparacao-descritores-detalhada) résume la précision globale obtenue par chaque descripteur.

Les résultats montrent que les performances du classificateur dépendent directement de la représentation choisie pour décrire les images.
Alors que l'utilisation directe des intensités des pixels est plus sensible aux dégradations introduites, les descripteurs LBP et HOG préservent mieux les informations pertinentes pour la classification, ce qui se traduit par de meilleures performances dans ce scénario. Il est important de souligner que les niveaux de bruit appliqués à chaque descripteur ont été choisis uniquement à des fins pédagogiques, afin d'illustrer le principe général selon lequel des descripteurs plus élaborés *peuvent* être plus robustes aux dégradations — ce qui ne signifie pas que cette relation se vérifie toujours, comme l'étude de cas de la section suivante le démontrera.

> ### 💡 Comment l'expérience est réalisée
>
> Comme l'objectif de cette section est de comparer uniquement l'effet des descripteurs,
> on génère un ensemble de données synthétiques simple : trois nuages de points
> gaussiens, centrés sur les mêmes valeurs de « couleur, texture et forme » déjà
> utilisées dans la [Figure 7.1](#fig-07-frutas-motivacao) — le même motif employé depuis le
> début du chapitre pour représenter les trois classes de fruits.
>
> ```python
> import numpy as np
> from sklearn.model_selection import train_test_split
> from sklearn.neighbors import KNeighborsClassifier
> from sklearn.metrics import accuracy_score, confusion_matrix
> ```
>
> ```python
> centros = {
>     "Maçã":    [0.8, 0.2, 0.9],
>     "Banana":  [0.3, 0.1, 0.2],
>     "Laranja": [0.9, 0.8, 0.8],
> }
>
> n_por_classe = 100
> X = np.vstack([
>     np.random.normal(centro, 0.12, (n_por_classe, 3))
>     for centro in centros.values()
> ])
> y = np.repeat(list(centros.keys()), n_por_classe)
> ```
>
> où :
>
> * `centros` : dictionnaire contenant le point moyen de chaque classe dans l'espace des caractéristiques (couleur, texture, forme) ;
> * `n_por_classe` : nombre d'échantillons générés par classe ;
> * `np.random.normal(centro, 0.12, (n_por_classe, 3))` : génère `n_por_classe` échantillons autour de chaque centre, avec un écart-type de 0,12 dans chaque dimension ;
> * `np.repeat(list(centros.keys()), n_por_classe)` : génère le vecteur d'étiquettes correspondant, dans le même ordre que les centres.
>
> Ensuite, on utilise le flux d'entraînement et d'évaluation :
>
> * `train_test_split(X, y, test_size=0.3)` : divise les données en entraînement (70 %) et test (30 %) ;
> * `KNeighborsClassifier(n_neighbors=5)` : crée un classificateur $k$-NN avec $k=5$ voisins ;
> * `fit(X_train, y_train)` : ajuste le modèle aux données d'entraînement ;
> * `predict(X_test)` : classe les échantillons de test ;
> * `accuracy_score(y_test, y_pred)` : calcule la précision ;
> * `confusion_matrix(y_test, y_pred)` : génère la matrice de confusion.
>
> Dans cette expérience, l'ensemble d'entraînement, le classificateur et la méthode
> d'évaluation restent exactement les mêmes. La seule différence entre les
> expériences réside dans la représentation utilisée pour chaque image (pixels
> bruts, LBP ou HOG), permettant d'évaluer exclusivement l'influence du
> descripteur sur la performance du classificateur.

In [7]:
classes = ["Maçã", "Banana", "Laranja"]

np.random.seed(42)

# Mêmes centres de classes (couleur, texture, forme) utilisés dans l'
# exemple précédent, maintenant réutilisés pour générer les données
# synthétiques d'entraînement et de test de cette expérience.
centros = {
    "Maçã":    [0.8, 0.2, 0.9],
    "Banana":  [0.3, 0.1, 0.2],
    "Laranja": [0.9, 0.8, 0.8],
}

n_por_classe = 100
X = np.vstack([
    np.random.normal(centro, 0.12, (n_por_classe, 3))
    for centro in centros.values()
])
y = np.repeat(list(centros.keys()), n_por_classe)

# Simuler des descripteurs avec différents niveaux de sensibilité au bruit.
# Plus le bruit ajouté est élevé, plus la représentation tend à être mauvaise.
descritores = {
    "Pixels Brutos": X + 0.5 * np.random.randn(*X.shape),
    "LBP":           X + 0.3 * np.random.randn(*X.shape),
    "HOG":           X + 0.2 * np.random.randn(*X.shape),
}

# Créer une seule figure avec 3 sous-graphiques côte à côte pour les matrices
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
resultados = {}

for idx, (nome, Xd) in enumerate(descritores.items()):
    X_train, X_test, y_train, y_test = train_test_split(Xd, y, test_size=0.3, random_state=42)
    
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    resultados[nome] = acc
    
    # Matrice de confusion dans le sous-graphique correspondant

    cm = confusion_matrix(y_test, y_pred, labels=classes)

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=classes,
        yticklabels=classes,
        ax=axes[idx],
    )

    axes[idx].set_title(
        f"{nome}\nAcurácia: {acc:.3f}",
        fontsize=11,
        fontweight="bold"
    )
    axes[idx].set_xlabel("Classe Predita")
    axes[idx].set_ylabel("Classe Real")


plt.tight_layout()
plt.show()


<Figure size 4200x1200 with 3 Axes>

**Figure 7.8:** Matrices de confusion obtenues par le classifieur k-NN utilisant trois descripteurs différents. Les lignes représentent la classe réelle (Pomme, Banane et Orange) et les colonnes la classe prédite. Plus la concentration de valeurs sur la diagonale principale est élevée, meilleure est la performance du descripteur.


In [8]:
# Comparaison visuelle sur une figure isolée
plt.figure(figsize=(6, 3.5))
nomes = list(resultados.keys())
acuracia = list(resultados.values())
colors = ['#6366f1', '#f97316', '#22c55e']

bars = plt.bar(nomes, acuracia, color=colors, width=0.5)
plt.ylabel('Acurácia Global')
plt.title('Desempenho Geral dos Descritores sob Ruído Realista', fontsize=12, fontweight='bold')
plt.ylim(0.5, 1.0)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Ajouter les valeurs au-dessus des barres en utilisant round standard pour l'affichage
for bar, val in zip(bars, acuracia):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

print("Analyse des résultats :")
print("- Pixels bruts : sensibles aux variations locales d'éclairage et au bruit.")
print("- LBP : bonne tolérance aux variations monotones de l'éclairage global.")
print("- HOG : excellent pour les contours et les formes stables sous de petites fluctuations géométriques.")

<Figure size 1800x1050 with 1 Axes>

**Figure 7.9:** Comparaison détaillée de l


Analyse des résultats :
- Pixels bruts : sensibles aux variations locales d'éclairage et au bruit.
- LBP : bonne tolérance aux variations monotones de l'éclairage global.
- HOG : excellent pour les contours et les formes stables sous de petites fluctuations géométriques.


## 7.9 Um Problema Concreto : Simulation de Descripteurs de Fruits

Reprenant le problème de classification des fruits présenté au début du chapitre, chaque image peut être représentée par un vecteur de caractéristiques (*feature vector*) obtenu à partir de l'extraction de descripteurs de couleur, de texture et de forme. La [Tableau 7.1](#tbl-descritores-frutas) présente quelques descripteurs fréquemment utilisés dans les applications de Vision par Ordinateur, y compris des techniques introduites dans le Chapitre 3 et dans ce chapitre.

<a id="tbl-descritores-frutas"></a>

**Tabela 7.1:** Ensemble de descripteurs chromatiques, texturaux et géométriques utilisés pour représenter des images de fruits.

| Caractéristique       | Description                                                |
|----------------------|----------------------------------------------------------|
| R, G, B              | intensité moyenne des canaux rouge, vert et bleu         |
| NC                   | intensité moyenne en niveaux de gris (*grayscale*)       |
| LBP                  | descripteur de texture (*Local Binary Pattern*)          |
| HOG                  | descripteur de forme (*Histogram of Oriented Gradients*) |
| Aire                 | nombre de pixels de l'objet                               |
| Périmètre            | longueur du contour                                      |
| Circularité          | mesure de la circularité de l'objet                      |
| Rapport largeur/hauteur | proportion entre la largeur et la hauteur de la région |


Dans cet exemple, chaque image est représentée par le vecteur

$$
X=(R,G,B,NC,\text{LBP},\text{HOG},\text{Área},\text{Perímetro},\text{Circularidade},\text{Razão}).
$$

> ### 📝 Simplification adoptée dans ce tableau
>
> En pratique, LBP et HOG ne sont pas des valeurs scalaires, mais des histogrammes comportant des dizaines ou des centaines de composantes. Dans cette section, chacun d'eux est représenté par une valeur unique uniquement pour simplifier la présentation. Dans les applications réelles, ces positions seraient remplacées par les composantes complètes des histogrammes respectifs.

Dans les applications réelles, tous les descripteurs ne contribuent pas de manière égale à distinguer les classes. Certains fournissent des informations plus pertinentes, tandis que d'autres peuvent être redondants ou peu discriminatifs.

Pour reproduire ce scénario de manière contrôlée, on utilisera `make_classification()`, de la bibliothèque `scikit-learn`. La fonction génère un ensemble de données synthétiques dont les caractéristiques peuvent être interprétées comme des descripteurs d'images, permettant de définir combien d'entre elles seront informatives pour la classification.

Dans cet exemple, dix caractéristiques synthétiques sont générées, dont seulement sept (`n_informative=7`) participent à la séparation entre les trois classes. Les autres simulent des attributs peu informatifs ou redondants. La [Figure 7.10](#fig-07-make-classification-ilustracao) présente une représentation conceptuelle de ce processus.

Avant d'introduire l'algorithme qui sera étudié en détail dans ce chapitre, il convient de faire une observation : pour identifier, parmi les dix caractéristiques synthétiques, lesquelles sont les plus discriminatives — et ainsi sélectionner deux d'entre elles pour la visualisation en 2D —, on utilise un *Random Forest* uniquement comme outil auxiliaire de diagnostic. Le KNN, objet de ce chapitre, est présenté ci-après.

In [9]:
# Configuration pour la reproduction
np.random.seed(42)

# Génération de données synthétiques avec des caractéristiques contrôlées
X, y = make_classification(
    n_samples=300,
    n_features=10,
    n_informative=7,
    n_redundant=2,
    n_repeated=1,        # Une caractéristique est une copie d'une autre
    n_classes=3,
    n_clusters_per_class=1,
    random_state=42,
)

# Créer une figure avec deux sous-graphiques
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Sous-graphique 1 : Visualisation des classes en 2D (en utilisant deux caractéristiques informatives)
# Identifier quelles caractéristiques sont les plus informatives
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)
importancias = rf.feature_importances_
caracteristicas_informativas = np.argsort(importancias)[-2:]  # Les deux plus importantes

cores = {0: "#e74c3c", 1: "#f1c40f", 2: "#e67e22"}  # Pomme, Banane, Orange
rotulos = {0: "Maçã", 1: "Banana", 2: "Laranja"}

for classe in range(3):
    idx = y == classe
    ax1.scatter(X[idx, caracteristicas_informativas[0]], 
               X[idx, caracteristicas_informativas[1]],
               c=cores[classe], label=rotulos[classe], 
               alpha=0.6, s=50, edgecolors="white", linewidth=0.5)

ax1.set_xlabel(f"Característica {caracteristicas_informativas[0]+1} (informativa)", fontsize=11)
ax1.set_ylabel(f"Característica {caracteristicas_informativas[1]+1} (informativa)", fontsize=11)
ax1.set_title("Classes no Espaço de Características\n(2 características informativas)", 
             fontsize=12, fontweight="bold")
ax1.legend(loc="upper right")
ax1.grid(alpha=0.3)

# Sous-graphique 2 : Importance des caractéristiques
bars = ax2.bar(range(1, 11), importancias, color="#4a90d9", alpha=0.7)
ax2.set_xlabel("Índice da Característica", fontsize=11)
ax2.set_ylabel("Importância", fontsize=11)
ax2.set_title("Importância de cada Característica\npara a Classificação", 
             fontsize=12, fontweight="bold")
ax2.set_xticks(range(1, 11))
ax2.grid(axis="y", alpha=0.3)

# Colorer les barres pour mettre en évidence les caractéristiques
cores_barras = ["#e74c3c" if i < 7 else "#95a5a6" for i in range(10)]
for bar, cor in zip(bars, cores_barras):
    bar.set_color(cor)

# Ajouter une légende
from matplotlib.patches import Patch
legenda_elements = [
    Patch(facecolor="#e74c3c", label="Características Informativas (7)"),
    Patch(facecolor="#95a5a6", label="Características Redundantes (3)")
]
ax2.legend(handles=legenda_elements, loc="upper right")

# Annoter le nombre de caractéristiques informatives
ax2.axhline(y=0.15, color="red", linestyle="--", alpha=0.3)
ax2.text(0.5, 0.17, "Limiar de importância", fontsize=9, color="red", alpha=0.7)

plt.tight_layout()
plt.show()

print("\n🔍 Analyse des données générées :")
print(f"  • Total d'échantillons : {X.shape[0]}")
print(f"  • Nombre de caractéristiques : {X.shape[1]}")
print(f"  • Caractéristiques informatives : 7 (colonnes 1 à 7 du graphique)")
print(f"  • Caractéristiques redondantes : 2 (colonnes 8 et 9)")
print(f"  • Caractéristiques répétées : 1 (colonne 10)")
print(f"  • Distribution des classes : {np.bincount(y)}")

<Figure size 4200x1500 with 2 Axes>

**Figure 7.10:** Illustration du processus de génération de données synthétiques avec *make_classification*. À gauche, visualisation des trois classes dans un espace bidimensionnel formé par deux caractéristiques informatives. À droite, importance relative de chaque caractéristique pour la classification, montrant que seulement 7 des 10 caractéristiques sont effectivement discriminantes, tandis que les autres sont redondantes (2) ou répétées (1).



🔍 Analyse des données générées :
  • Total d'échantillons : 300
  • Nombre de caractéristiques : 10
  • Caractéristiques informatives : 7 (colonnes 1 à 7 du graphique)
  • Caractéristiques redondantes : 2 (colonnes 8 et 9)
  • Caractéristiques répétées : 1 (colonne 10)
  • Distribution des classes : [101  98 101]


## 7.10 Classificateur k-NN : Comment il fonctionne en interne

Le **k-*Nearest Neighbors* (k-NN)** est l'un des algorithmes de classification les plus simples et intuitifs de l'apprentissage automatique. Contrairement à de nombreux classificateurs, il ne construit pas explicitement un modèle pendant la phase d'entraînement. Au lieu de cela, il stocke les échantillons étiquetés et, lorsqu'un nouvel échantillon doit être classifié, il recherche ceux qui lui ressemblent le plus.

Le principe de l'algorithme repose sur l'hypothèse que des échantillons présentant des caractéristiques similaires ont tendance à appartenir à la même classe. Pour quantifier cette proximité, le k-NN utilise une mesure de distance entre les vecteurs de caractéristiques.

À titre d'exemple, considérons la [Tableau 7.2](#tbl-knn-frutas), qui présente une version simplifiée du problème de classification des fruits utilisant seulement deux caractéristiques : l'intensité de la couleur et la circularité, toutes deux normalisées dans l'intervalle de 0 à 1.

<a id="tbl-knn-frutas"></a>

**Tabela 7.2:** Exemple simplifié de classification de fruits utilisant deux caractéristiques normalisées.

| Échantillon      | Couleur | Circularité | Classe |
|------------------|--------:|------------:|--------|
| Fruit 1          | 0,82    | 0,88        | Pomme  |
| Fruit 2          | 0,30    | 0,20        | Banane |
| Fruit 3          | 0,88    | 0,85        | Pomme  |
| Fruit ? (test)   | 0,80    | 0,90        | ?      |


En observant uniquement ces deux caractéristiques, on remarque que le fruit de test est beaucoup plus proche des échantillons étiquetés comme **Pomme** que de l'échantillon étiqueté comme **Banane**. Dans la section suivante, cette notion intuitive de proximité sera formalisée au moyen d'une métrique de distance, utilisée par l'algorithme pour identifier les voisins les plus proches et décider de la classe du nouvel échantillon.

### 7.10.1 Métrique de distance

La proximité entre deux échantillons est généralement quantifiée par la
**distance euclidienne**, définie par

$$
d(x,x_i)=\|x-x_i\|_2=
\sqrt{\sum_{j=1}^{n}(x_j-x_{i,j})^2},
$$

où :

-   $x$ est l’échantillon de test ;
-   $x_i$ est un échantillon de l’ensemble d’entraînement ;
-   $n$ est le nombre de caractéristiques ;
-   $x_j$ et $x_{i,j}$ représentent la $j$-ième caractéristique.

Dans l’implémentation de ce chapitre, $x$ correspond à une ligne de `X_test`
et $x_i$ à une ligne de `X_train`. La méthode `predict()` calcule
automatiquement la distance entre $x$ et tous les échantillons
d’entraînement.

Dans l’exemple de la [Tableau 7.2](#tbl-knn-frutas):

$$
d(\text{teste}, \text{Fruta 1}) \approx 0{,}028,\qquad
d(\text{teste}, \text{Fruta 2}) \approx 0{,}860,\qquad
d(\text{teste}, \text{Fruta 3}) \approx 0{,}094.
$$

Comme les plus petites distances correspondent aux Fruits 1 et 3, ces échantillons
seront utilisés lors de l’étape de décision.

### 7.10.2 Règle de décision

Après avoir trié les distances, l’algorithme sélectionne les $k$ voisins les plus proches. Soit $N_k(x)$ cet ensemble. La classe prédite est donnée par

$$
\hat y=\operatorname{moda}\{\,y_i:x_i\in N_k(x)\,\},
$$

où $y_i$ est le label de l’échantillon $x_i$ et $\hat y$ est la classe attribuée à l’échantillon de test.

Dans l’exemple, pour $k=3$, les voisins sont Fruit 1 (Pomme), Fruit 3 (Pomme) et Fruit 2 (Banane). Comme **Pomme** reçoit deux votes, c’est la classe prédite.

::: callout-tip
### Classe `KNeighborsClassifier`

Dans ce chapitre, l'algorithme est implémenté avec la classe
`KNeighborsClassifier` de la bibliothèque `scikit-learn` :

``` python
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)
```

où :

-   `KNeighborsClassifier(n_neighbors=3)` : définit la valeur de $k$ ;
-   `fit(X_train, y_train)` : stocke les échantillons d'entraînement
    (`X_train`) et leurs étiquettes (`y_train`) ;
-   `predict(X_test)` : retourne les classes prédites pour les échantillons de
    `X_test`.

En interne, `predict()` exécute les étapes décrites précédemment :
calcule les distances, identifie les $k$ voisins les plus proches et
détermine la classe par vote majoritaire.
:::

La [Figure 7.11](#fig-07-knn-passo-passo) illustre cette procédure sur un ensemble
bidimensionnel. La figure met en évidence les voisins utilisés lors de la classification,
tandis que la console présente les étapes de l'algorithme : calcul des
distances, tri, sélection des voisins, vote et prédiction de la
classe.

In [10]:
def knn_passo_a_passo(X, y, x, k=3):
    """Exécute les cinq étapes de l’algorithme k-NN.
    Paramètres : X (entraînement), y (étiquettes), x (test) et k (nombre de voisins).
    """
    # 1. Distances
    dist = [(np.linalg.norm(x-xi), yi, i) for i, (xi, yi) in enumerate(zip(X, y))]
    print(f"1. Distances calculées : {len(dist)}")

    # 2. Tri
    dist.sort(key=lambda t: t[0])
    print("2. Distances triées")

    # 3. Sélection
    vizinhos = dist[:k]
    print(f"3. {k} voisins les plus proches :")
    for d, c, _ in vizinhos:
        print(f"   {d:.4f} → {c}")

    # 4. Vote
    votos = {}
    for _, c, _ in vizinhos:
        votos[c] = votos.get(c, 0) + 1
    print("4. Votes :", votos)

    # 5. Décision
    classe = max(votos, key=votos.get)
    print("5. Classe prédite :", classe)

    return classe, vizinhos


# Données d’exemple
np.random.seed(4)
X = np.r_[np.random.randn(15,2)+[2,2],
          np.random.randn(15,2)+[-2,-2]]
y = np.array(["Classe A"]*15 + ["Classe B"]*15)
x = np.array([0.5,0.5])

classe, vizinhos = knn_passo_a_passo(X, y, x)

# Visualisation
plt.figure(figsize=(5,5))

for c, rotulo in [("Classe A","Classe A"), ("Classe B","Classe B")]:
    P = X[y==c]
    plt.scatter(P[:,0], P[:,1], s=80, label=rotulo)

plt.scatter(*x, marker="*", s=220, edgecolors="black", label="Teste")

for _, _, i in vizinhos:
    plt.scatter(*X[i], s=220, facecolors="none", edgecolors="black", linewidths=2)
    plt.plot([x[0], X[i,0]], [x[1], X[i,1]], "--", lw=1)

plt.xlabel("Característica 1")
plt.ylabel("Característica 2")
plt.title(f"k-NN ($k=3$): classe predita = {classe}")
plt.legend()
plt.grid(alpha=.3)
plt.axis("equal")
plt.tight_layout()
plt.show()

1. Distances calculées : 30
2. Distances triées
3. 3 voisins les plus proches :
   1.0850 → Classe A
   1.6379 → Classe B
   1.6382 → Classe A
4. Votes : {np.str_('Classe A'): 2, np.str_('Classe B'): 1}
5. Classe prédite : Classe A


<Figure size 1500x1500 with 1 Axes>

**Figure 7.11:** Classification d’un nouvel échantillon par l’algorithme k-NN. Le point de test (étoile) est classé à partir des trois voisins les plus proches, mis en évidence par des cercles.


### 7.10.3 Le rôle du paramètre $k$

Le paramètre $k$ détermine combien de voisins participent à la décision de classification.

- **Les petites valeurs de $k$** (par exemple, $k=1$) rendent le classificateur plus sensible aux bruits et aux variations locales, produisant des frontières de décision plus irrégulières et favorisant le *surapprentissage* (*overfitting*).
- **Les grandes valeurs de $k$** produisent des frontières de décision plus lisses, mais peuvent réduire la sensibilité aux structures locales, favorisant le *sous-apprentissage* (*underfitting*).

Dans les problèmes à deux classes, il est courant d’utiliser des valeurs impaires de $k$ afin de réduire l’occurrence d’égalités.

Un autre aspect important est la **malédiction de la dimensionnalité** (*curse of dimensionality*). À mesure que le nombre de caractéristiques augmente, les distances entre les échantillons tendent à devenir plus similaires, rendant difficile l’identification de voisins réellement représentatifs.

::: callout-note
### Résumé

L’algorithme k-NN peut être résumé en trois étapes :

1. extraire le vecteur de caractéristiques du nouvel échantillon ;
2. identifier les $k$ voisins les plus proches ;
3. classer l’échantillon selon la classe la plus fréquente parmi ces voisins.

Le simulateur de la [Figure 7.12](#fig-07-sim-07-knn) permet d’explorer visuellement l’effet du paramètre $k$ sur la frontière de décision.
:::

In [11]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-07-knn" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-07-knn * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-07-knn canvas { display: block; max-width: 100%; height: auto; border-radius: 8px; border: 1px solid #e4dcc8; background: #ffffff; cursor: crosshair; margin: 0 auto; }
  #sim-07-knn button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-07-knn button:hover { background: #e8dfcf; }
  #sim-07-knn button.sim-07-knn_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  #sim-07-knn input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-07-knn_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-07-knn_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-07-knn_grid_stats { display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; margin-bottom: 12px; }
  .sim-07-knn_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; }
  .sim-07-knn_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .sim-07-knn_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  .sim-07-knn_legend { display: flex; align-items: center; gap: 6px; font-size: 11px; font-weight: 600; color: #5e5a4a; }
  .sim-07-knn_dot { width: 10px; height: 10px; border-radius: 50%; display: inline-block; border: 1px solid rgba(38,36,29,0.2); }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">📌 Simulateur : Frontière de décision du k-NN</span>
  <span class="sim-07-knn_pill">Cliquez sur le canvas pour ajouter des points</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles e Estatísticas -->
  <div class="sim-07-knn_panel" style="margin-bottom:14px;">
    
    <div class="sim-07-knn_grid_stats">
      <div class="sim-07-knn_stat_box">
        <div class="sim-07-knn_stat_label">Voisins (k)</div>
        <div id="sim-07-knn_kVal" class="sim-07-knn_stat_value" style="color:#2980b9;">3</div>
      </div>
      <div class="sim-07-knn_stat_box">
        <div class="sim-07-knn_stat_label">Classe bleue</div>
        <div id="sim-07-knn_nAzul" class="sim-07-knn_stat_value" style="color:#2980b9;">0</div>
      </div>
      <div class="sim-07-knn_stat_box">
        <div class="sim-07-knn_stat_label">Classe rouge</div>
        <div id="sim-07-knn_nVerm" class="sim-07-knn_stat_value" style="color:#c0392b;">0</div>
      </div>
    </div>

    <!-- Botões de Ação -->
    <div style="display:flex; gap:6px; flex-wrap:wrap; margin-bottom:12px; justify-content:center;">
      <button id="sim-07-knn_btnAzul" class="sim-07-knn_active">🔵 Ajouter bleu</button>
      <button id="sim-07-knn_btnVerm">🔴 Ajouter rouge</button>
      <button id="sim-07-knn_btnLimpar" style="border-color:#f5b7b1; color:#c0392b; background:#fdecea;">🗑️ Effacer</button>
      <button id="sim-07-knn_btnReset">↺ Réinitialiser</button>
    </div>

    <!-- Slider k -->
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Valeur de k : <span id="sim-07-knn_slVal" style="font-family:monospace; color:#26241d;">3</span>
      </label>
    </div>
    <input type="range" id="sim-07-knn_slider" min="1" max="15" step="1" value="3">

  </div>

  <!-- Canvas -->
  <div style="margin-bottom:14px; text-align:center;">
    <canvas id="sim-07-knn_canvas" width="640" height="360"></canvas>
  </div>

  <!-- Legenda e Rodapé -->
  <div class="sim-07-knn_panel">
    <div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;">
      <div style="display:flex; gap:16px;">
        <div class="sim-07-knn_legend"><span class="sim-07-knn_dot" style="background:#2980b9;"></span> Classe bleue</div>
        <div class="sim-07-knn_legend"><span class="sim-07-knn_dot" style="background:#c0392b;"></span> Classe rouge</div>
      </div>
      <div style="font-size:10.5px; color:#8a8371; font-weight:600;">
        La région colorée de fond représente la classe attribuée par l'algorithme à chaque point de l'espace.
      </div>
    </div>
  </div>

</div>
</div>

<script>
(function(){
  function initSim07Knn(root){
    if (!root || root.dataset.sim07KnnInit) return;
    root.dataset.sim07KnnInit = "1";

    const canvas = root.querySelector("#sim-07-knn_canvas");
    const ctx = canvas.getContext("2d");
    const W = canvas.width, H = canvas.height;
    const CELL = 10;

    let currentLabel = 0; // 0 = azul, 1 = vermelho
    let kVal = 3;
    let points = [];

    function defaultPoints(){
      return [
        {x:120,y:260,l:0},{x:150,y:230,l:0},{x:100,y:200,l:0},{x:160,y:290,l:0},
        {x:90,y:250,l:0},{x:140,y:190,l:0},{x:175,y:250,l:0},{x:110,y:150,l:0},
        {x:200,y:230,l:0},{x:130,y:310,l:0},
        {x:480,y:100,l:1},{x:450,y:130,l:1},{x:500,y:160,l:1},{x:440,y:80,l:1},
        {x:510,y:110,l:1},{x:470,y:60,l:1},{x:430,y:150,l:1},{x:520,y:190,l:1},
        {x:490,y:220,l:1},{x:460,y:190,l:1},
        {x:300,y:170,l:0},{x:320,y:190,l:1},{x:280,y:200,l:1},{x:310,y:150,l:0}
      ];
    }

    function classify(x, y, k, pontos){
      if (pontos.length === 0) return null;
      const dists = pontos.map(p => ({
        d: (p.x - x) * (p.x - x) + (p.y - y) * (p.y - y),
        l: p.l
      }));
      dists.sort((a, b) => a.d - b.d);
      const vizinhos = dists.slice(0, Math.min(k, dists.length));
      let votos = [0, 0];
      vizinhos.forEach(v => votos[v.l]++);
      return votos[1] > votos[0] ? 1 : 0;
    }

    function render(){
      ctx.clearRect(0, 0, W, H);

      // Região de decisão em grade
      for (let gy = 0; gy < H; gy += CELL){
        for (let gx = 0; gx < W; gx += CELL){
          const cx = gx + CELL / 2, cy = gy + CELL / 2;
          const classe = points.length > 0 ? classify(cx, cy, kVal, points) : null;
          if (classe === 0) ctx.fillStyle = "rgba(41, 128, 185, 0.15)";
          else if (classe === 1) ctx.fillStyle = "rgba(192, 57, 43, 0.15)";
          else ctx.fillStyle = "#fafaf7";
          ctx.fillRect(gx, gy, CELL, CELL);
        }
      }

      // Desenhar pontos de treinamento
      points.forEach(p => {
        ctx.beginPath();
        ctx.arc(p.x, p.y, 6.5, 0, 2 * Math.PI);
        ctx.fillStyle = p.l === 0 ? "#2980b9" : "#c0392b";
        ctx.fill();
        ctx.lineWidth = 1.5;
        ctx.strokeStyle = "#ffffff";
        ctx.stroke();
      });

      root.querySelector("#sim-07-knn_nAzul").textContent = points.filter(p => p.l === 0).length;
      root.querySelector("#sim-07-knn_nVerm").textContent = points.filter(p => p.l === 1).length;
      root.querySelector("#sim-07-knn_kVal").textContent = kVal;
    }

    canvas.addEventListener("click", function(ev){
      const rect = canvas.getBoundingClientRect();
      const scaleX = canvas.width / rect.width;
      const scaleY = canvas.height / rect.height;
      const x = (ev.clientX - rect.left) * scaleX;
      const y = (ev.clientY - rect.top) * scaleY;
      points.push({x: x, y: y, l: currentLabel});
      render();
    });

    root.querySelector("#sim-07-knn_slider").addEventListener("input", function(ev){
      kVal = parseInt(ev.target.value, 10);
      root.querySelector("#sim-07-knn_slVal").textContent = kVal;
      render();
    });

    const btnAzul = root.querySelector("#sim-07-knn_btnAzul");
    const btnVerm = root.querySelector("#sim-07-knn_btnVerm");

    function setLabel(l){
      currentLabel = l;
      btnAzul.classList.toggle("sim-07-knn_active", l === 0);
      btnVerm.classList.toggle("sim-07-knn_active", l === 1);
    }

    btnAzul.addEventListener("click", function(){ setLabel(0); });
    btnVerm.addEventListener("click", function(){ setLabel(1); });

    root.querySelector("#sim-07-knn_btnLimpar").addEventListener("click", function(){
      points = [];
      render();
    });

    root.querySelector("#sim-07-knn_btnReset").addEventListener("click", function(){
      points = defaultPoints();
      render();
    });

    setLabel(0);
    points = defaultPoints();
    render();
  }

  function tryInitSim07Knn(){
    var root = document.getElementById("sim-07-knn");
    if (root) initSim07Knn(root); else setTimeout(tryInitSim07Knn, 200);
  }
  tryInitSim07Knn();
})();
</script>
</div>
""")

**Figure 7.12:** Simulateur interactif de la frontière de décision du k-NN : ajoutez des points d


<figure id="fig-07-sim-07-knn">
  <img src="imagens/fig-07-sim-07-knn.png" alt=" Simulateur interactif de la frontière de décision du k-NN : ajoutez des points d'entraînement et ajustez la valeur de k pour observer l'effet sur la région de décision. " style="max-width:80%" />
  <figcaption><strong>Figure 7.12:</strong>  Simulateur interactif de la frontière de décision du k-NN : ajoutez des points d'entraînement et ajustez la valeur de k pour observer l'effet sur la région de décision. </figcaption>
</figure>

## 7.11 Projet Pratique 1 : Classification de Chiffres Manuscrits avec k-NN

Les sections précédentes ont présenté l’algorithme k-NN à travers un exemple simplifié de classification de fruits, n’utilisant que deux caractéristiques. Ci-après, le même algorithme est appliqué à un ensemble de données d’images, dans lequel chaque échantillon est représenté par un vecteur de plus grande dimension.

Comme étude de cas, on utilise la base publique `load_digits`, mise à disposition par la bibliothèque `scikit-learn`. Cet ensemble de données contient 1797 images de chiffres manuscrits des classes de 0 à 9, chacune ayant une résolution de $8 \times 8$ pixels en niveaux de gris. Chaque image est représentée par un vecteur de 64 caractéristiques, correspondant aux intensités des pixels, et chaque vecteur possède une étiquette indiquant le chiffre correspondant.

La base `load_digits` est mise à disposition par la bibliothèque `scikit-learn` et est utilisée dans ce chapitre pour illustrer l’application de l’algorithme k-NN. En plus d’être disponible directement dans `scikit-learn`, elle dispense d’étapes supplémentaires d’obtention et de préparation des données, ce qui permet de concentrer l’attention sur l’implémentation et l’évaluation du classifieur.

La [Figure 7.13](#fig-07-digits-amostra) présente un échantillon des images de la base de données.

In [12]:
digits = load_digits()
print(f"Total d'échantillons : {digits.data.shape[0]}, dimension du vecteur : {digits.data.shape[1]}")
print(f"Classes : {[int(i) for i in sorted(set(digits.target))]}")

n_amostras = 16
imgs = list(digits.images[:n_amostras])
imgs_titles = [str(label) for label in digits.target[:n_amostras]]
mm.show(imgs, titles=imgs_titles, cols=8, figsize=(12, 4))


Total d'échantillons : 1797, dimension du vecteur : 64
Classes : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


<Figure size 1800x600 with 16 Axes>

**Figure 7.13:** Échantillon de chiffres manuscrits de la base *load_digits*, utilisé comme étude de cas de classification.


### 7.11.1 Classification avec des vecteurs d’intensité

Dans cette première expérience, chaque image de dimension $8 \times 8$ est représentée directement par les intensités de ses 64 pixels, sans extraction de descripteurs supplémentaires. Ainsi, chaque échantillon correspond à un vecteur de 64 caractéristiques, utilisé comme entrée du classifieur k-NN.

Ensuite, l’ensemble de données est divisé en sous-ensembles d’entraînement et de test, en préservant la proportion des dix classes au moyen du paramètre `stratify=y`. Le classifieur est entraîné avec $k=3$ et évalué sur l’ensemble de test en utilisant la précision et la matrice de confusion présentée dans la [Figure 7.14](#fig-07-knn-pixels)..

In [13]:
X, y = digits.data, digits.target

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

n_neighbors = 3
knn_pixels = KNeighborsClassifier(n_neighbors=n_neighbors)
knn_pixels.fit(X_treino, y_treino)
pred_pixels = knn_pixels.predict(X_teste)

acc_pixels = accuracy_score(y_teste, pred_pixels)
print(f"Précision (vecteurs d'intensité, k={n_neighbors}): {acc_pixels:.4f}")

cm = confusion_matrix(y_teste, pred_pixels)

plt.figure(figsize=(5,4))
plt.imshow(cm, cmap="Blues")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(
            j, i, cm[i, j],
            ha="center", va="center",
            color="white" if cm[i, j] > cm.max()/2 else "black",
            fontsize=9
        )

plt.title("Matriz de Confusão — Pixels Brutos")
plt.xlabel("Classe Predita")
plt.ylabel("Classe Real")
plt.xticks(range(10))
plt.yticks(range(10))
plt.colorbar(fraction=0.046)
plt.tight_layout()
plt.show()


Précision (vecteurs d'intensité, k=3): 0.9870


<Figure size 1500x1200 with 2 Axes>

**Figure 7.14:** Matrice de confusion du classificateur k-NN entraîné avec des vecteurs d


### 7.11.2 Classification avec des descripteurs HOG

Dans l’expérience précédente, chaque image était représentée directement par les intensités de ses pixels. Dans cette section, cette représentation est remplacée par des descripteurs HOG (*Histogram of Oriented Gradients*), qui encodent des informations sur la distribution des orientations des gradients de l’image.

On conserve la même partition des données, le même classificateur k-NN et le même protocole d’évaluation, en modifiant uniquement la représentation des images. La [Figure 7.15](#fig-07-comparativo-descritores) compare les résultats obtenus avec des vecteurs d’intensité et avec des descripteurs HOG.

In [14]:
descritores_hog = np.array([
    hog(img, orientations=8, pixels_per_cell=(4, 4), cells_per_block=(1, 1))
    for img in digits.images
])
print(f"Dimension du vecteur HOG : {descritores_hog.shape[1]}")

Xh_treino, Xh_teste, yh_treino, yh_teste = train_test_split(
    descritores_hog, y, test_size=0.3, random_state=42, stratify=y
)

n_neighbors = 3
knn_hog = KNeighborsClassifier(n_neighbors=n_neighbors)
knn_hog.fit(Xh_treino, yh_treino)
pred_hog = knn_hog.predict(Xh_teste)
acc_hog = accuracy_score(yh_teste, pred_hog)
print(f"Précision (descripteur HOG, k={n_neighbors}) : {acc_hog:.4f}")

plt.figure(figsize=(4, 3))
plt.bar(["Pixels brutos", "HOG"], [acc_pixels, acc_hog], color=["#6366f1", "#f97316"])
plt.ylim(0, 1.1)  # Augmente la limite supérieure pour laisser de l'espace
plt.ylabel("Acurácia")
plt.title("Comparação de Descritores")
for i, v in enumerate([acc_pixels, acc_hog]):
    plt.text(i, v + 0.02, f"{v:.3f}", ha="center")  # Augmente le décalage vertical
plt.tight_layout()

Dimension du vecteur HOG : 32
Précision (descripteur HOG, k=3) : 0.7593


<Figure size 1200x900 with 1 Axes>

**Figure 7.15:** Comparaison de précision entre les descripteurs de pixels bruts et HOG pour le classificateur k-NN (k=3) sur la base de chiffres.


> ### 📝 Pourquoi cela se produit-il ?
>
> Dans la [Figure 7.15](#fig-07-comparativo-descritores), le classifieur entraîné avec des vecteurs d'**intensité des pixels** atteint une précision plus élevée ($0.987$) que celui basé sur des descripteurs **HOG** ($0.759$). Ce résultat est lié aux caractéristiques de la base `load_digits`.
>
> Les images ont une résolution de seulement $8\times8$ pixels, sont approximativement centrées et présentent peu de variation d'éclairage, d'échelle et d'orientation. Dans ce scénario, les intensités des pixels préservent pratiquement toute l'information nécessaire pour distinguer les classes. En revanche, le HOG résume l'image en histogrammes d'orientations des gradients, réduisant une partie du détail spatial disponible dans les pixels d'origine.
>
> Cette réduction d'information peut rendre difficile la séparation de chiffres visuellement similaires, comme 3 et 8 ou 4 et 9, surtout lorsque la résolution de l'image est faible.
>
> Dans des problèmes avec des images de plus haute résolution ou sujettes à des variations d'éclairage, de position, d'échelle ou de petites déformations, des descripteurs comme le HOG tendent à mieux représenter la structure locale de l'image que les valeurs individuelles des pixels. Ainsi, cette expérience illustre un principe important de l'apprentissage automatique : **la représentation des données doit être choisie en fonction des caractéristiques du problème et non de la complexité du descripteur.**

## 7.12 Évaluation des Classifieurs

Dans les sections précédentes, la qualité du classifieur a été analysée à
l’aide de l’exactitude et de la matrice de confusion. Dans cette section,
ces outils sont complétés par des métriques utilisées pour l’évaluation
des modèles et par une procédure permettant de sélectionner la valeur du
paramètre $k$.

L’exactitude correspond à la proportion d’échantillons correctement
classifiés. Bien qu’elle soit une mesure simple et largement utilisée,
elle peut s’avérer insuffisante lorsque les classes présentent des
distributions très déséquilibrées.

À partir de la matrice de confusion — introduite dans le **Chapitre 1** et
utilisée tout au long de ce chapitre — des métriques par classe peuvent
être calculées, comme la précision et le rappel :

$$
\text{Précision}=\frac{VP}{VP+FP},
\qquad
\text{Rappel}=\frac{VP}{VP+FN},
$$

où $VP$, $FP$ et $FN$ représentent, respectivement, le nombre de vrais
positifs, de faux positifs et de faux négatifs de la classe analysée.
La précision quantifie la proportion de prédictions positives correctes,
tandis que le rappel mesure la capacité du classifieur à identifier les
exemples appartenant à la classe.

### 7.12.1 Choix de $k$ par Validation Croisée

Dans les expériences précédentes, on a adopté $k=3$ pour illustrer le fonctionnement de l'algorithme. Cependant, ce paramètre influence directement la performance du classificateur et, en pratique, doit être sélectionné à partir des données.

Une approche largement utilisée est la **validation croisée** (*cross-validation*), dans laquelle l'ensemble d'entraînement est divisé en partitions successives pour estimer la performance du modèle sur des données non utilisées pendant l'entraînement.

Le code suivant calcule la précision moyenne obtenue par validation croisée de cinq partitions (*5-fold cross-validation*) pour différentes valeurs de $k$. La [Figure 7.16](#fig-07-elbow-k) présente les résultats, permettant d'identifier la région où le classificateur atteint la meilleure performance.

In [15]:
valores_k = range(1, 16)
acuracias_medias = []

for k in valores_k:
    modelo = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(modelo, X, y, cv=5)
    acuracias_medias.append(scores.mean())

melhor_k = list(valores_k)[int(np.argmax(acuracias_medias))]
print(f"Meilleure valeur de k trouvée : {melhor_k} (précision moyenne={max(acuracias_medias):.4f})")

plt.figure(figsize=(6, 4))
plt.plot(list(valores_k), acuracias_medias, marker="o", color="#4f46e5")
plt.axvline(melhor_k, color="#f97316", linestyle="--", label=f"melhor k = {melhor_k}")
plt.xlabel("k")
plt.ylabel("Acurácia média (validação cruzada)")
plt.title("Seleção de k por Validação Cruzada")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

Meilleure valeur de k trouvée : 2 (précision moyenne=0.9672)


<Figure size 1800x1200 with 1 Axes>

**Figure 7.16:** Précision moyenne par validation croisée (5 partitions) en fonction du paramètre k, pour la base de chiffres avec vecteurs d


### 7.12.2 Le compromis entre biais et variance : Diagnostic de *surajustement* et de *sous-apprentissage*

L'hyperparamètre $k$ influence la complexité de la frontière de décision du classificateur k-NN et, par conséquent, sa capacité de généralisation. En termes généraux, de petites valeurs de $k$ rendent le modèle plus sensible aux échantillons d'entraînement, tandis que des valeurs plus grandes produisent des frontières de décision plus lisses.

Ces comportements sont associés au compromis entre **biais** (*bias*) et **variance** (*variance*). De très petites valeurs de $k$ tendent à augmenter le risque de **surajustement** (*overfitting*), surtout dans des ensembles de données bruités, alors que des valeurs très grandes peuvent conduire au **sous-apprentissage** (*underfitting*), réduisant la capacité du modèle à capturer les structures locales des données.

Alors que la [Figure 7.16](#fig-07-elbow-k) n'a présenté que la précision moyenne obtenue par validation croisée, la [Figure 7.17](#fig-07-overfitting-analysis) compare les précisions d'entraînement et de test pour différentes valeurs de $k$. Les régions mises en évidence sur le graphique représentent le comportement attendu de l'algorithme : un risque plus élevé de surajustement pour de petites valeurs de $k$, une région intermédiaire qui produit souvent un bon équilibre entre biais et variance, et un risque plus élevé de sous-apprentissage pour de grandes valeurs de $k$.

Cependant, ces régions doivent être interprétées uniquement comme une référence conceptuelle. Le comportement observé dépend des caractéristiques de l'ensemble de données. Sur la base `load_digits`, par exemple, les images présentent peu de variabilité et une bonne séparation entre les classes, de sorte que de petites valeurs de $k$ peuvent afficher des performances similaires — voire supérieures — aux autres, sans montrer un surajustement significatif.

In [16]:
k_values = range(1, 16)
train_acc, test_acc = [], []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k).fit(X_treino, y_treino)
    train_acc.append(accuracy_score(y_treino, knn.predict(X_treino)))
    test_acc.append(accuracy_score(y_teste, knn.predict(X_teste)))

plt.figure(figsize=(9,5))
plt.plot(k_values, train_acc, "o-", lw=2, label="Treinamento")
plt.plot(k_values, test_acc,  "s-", lw=2, label="Teste")

plt.axvspan(1, 3,  color="#fca5a5", alpha=.25, label="Maior risco de overfitting")
plt.axvspan(3,11,  color="#86efac", alpha=.25, label="Compromisso entre viés e variância")
plt.axvspan(11,15, color="#93c5fd", alpha=.25, label="Maior risco de underfitting")

plt.xlabel("Número de vizinhos ($k$)")
plt.ylabel("Acurácia")
plt.xticks(k_values)
plt.grid(alpha=.3)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

maior_acc = max(test_acc)
melhores_k = [k for k, a in zip(k_values, test_acc) if np.isclose(a, maior_acc)]

print("Interprétation")
print("- Petites valeurs de k : risque plus élevé de surajustement.")
print("- Valeurs intermédiaires : meilleur compromis entre biais et variance.")
print("- Grandes valeurs de k : risque plus élevé de sous-ajustement.")
print("\nLes régions colorées représentent des tendances générales ;")
print("le comportement observé dépend de l'ensemble de données.")
print(f"\nPrécision maximale sur le test : {maior_acc:.3f}")
print(f"Valeurs de k ayant atteint cette précision : {melhores_k}")

<Figure size 2700x1500 with 1 Axes>

**Figure 7.17:** Précision sur les ensembles d


Interprétation
- Petites valeurs de k : risque plus élevé de surajustement.
- Valeurs intermédiaires : meilleur compromis entre biais et variance.
- Grandes valeurs de k : risque plus élevé de sous-ajustement.

Les régions colorées représentent des tendances générales ;
le comportement observé dépend de l'ensemble de données.

Précision maximale sur le test : 0.987
Valeurs de k ayant atteint cette précision : [1, 2, 3, 5]


## 7.13 Projet Pratique 2 : Comparaison de Descripteurs pour la Classification de Textures

Dans le **Chapitre 6**, la variance locale a été utilisée comme descripteur de texture pour **détecter** des anomalies sur des surfaces industrielles, en distinguant des échantillons **conformes** et **défectueux**. Dans ce projet, le problème est reformulé comme une tâche de **classification multiclasse**, dans laquelle différentes représentations de l'image sont utilisées comme entrée pour un classificateur.

Trois types de descripteurs seront considérés : les **intensités des pixels**, le **Local Binary Patterns (LBP)** et l'**Histogram of Oriented Gradients (HOG)**. Pour chaque représentation, un vecteur de caractéristiques sera extrait et servira d'entrée à l'algorithme des $k$ plus proches voisins ($k$-NN). À la fin, les précisions obtenues par chaque descripteur dans les conditions définies pour cette expérience seront comparées.

L'évaluation sera réalisée par **validation croisée stratifiée en cinq partitions (*5-fold stratified cross-validation*)**. Dans cette procédure, l'ensemble de données est divisé en cinq sous-ensembles en préservant la proportion entre les classes. À chaque itération, une partition est utilisée pour le test et les quatre restantes pour l'entraînement, le processus étant répété jusqu'à ce que toutes les partitions aient été utilisées comme ensemble de test. À l'issue des cinq exécutions, la précision moyenne et l'écart-type sont calculés pour chaque descripteur.

L'ensemble de données est composé de trois classes de textures synthétiques : **granulaire**, obtenue à partir de bruit gaussien lissé ; **rayée**, formée par des motifs sinusoïdaux périodiques ; et **tachetée**, composée de régions circulaires superposées. Pour introduire de la variabilité entre les échantillons, toutes les images reçoivent une perturbation par bruit gaussien de faible intensité. La [Figure 7.18](#fig-07-texturas-amostra) présente des exemples des trois classes utilisées dans l'expérience.

In [17]:
rng = np.random.default_rng(42)

def gerar_textura(classe, tamanho=64, ruido=0.10):
    """Génère une texture synthétique 64x64 appartenant à l'une des trois classes."""
    if classe == "granular":
        escala = rng.uniform(0.14, 0.22)
        img = rng.normal(0.5, escala, (tamanho, tamanho))
        img = cv2.GaussianBlur(img.astype(np.float32), (3, 3), 0)

    elif classe == "listrada":
        n_periodos = rng.uniform(4, 8)
        amplitude = rng.uniform(0.22, 0.38)
        eixo_x = np.linspace(0, n_periodos * np.pi, tamanho)
        base = 0.5 + amplitude * np.sin(eixo_x)
        img = np.tile(base, (tamanho, 1)).astype(np.float32)
        img += rng.normal(0, 0.09, (tamanho, tamanho)).astype(np.float32)

    elif classe == "manchada":
        img = np.full((tamanho, tamanho), 0.5, dtype=np.float32)
        n_manchas = rng.integers(5, 11)
        for _ in range(n_manchas):
            cx, cy = rng.integers(0, tamanho, 2)
            raio = int(rng.integers(3, 11))
            intensidade = float(rng.uniform(0.15, 0.9))
            cv2.circle(img, (int(cx), int(cy)), raio, intensidade, -1)
        img = cv2.GaussianBlur(img, (5, 5), 0)

    else:
        raise ValueError(f"Classe desconhecida: {classe}")

    img = img + rng.normal(0, ruido, (tamanho, tamanho)).astype(np.float32)
    img = np.clip(img, 0, 1)
    return (img * 255).astype(np.uint8)

classes_textura = ["granular", "listrada", "manchada"]
amostras = [gerar_textura(c) for c in classes_textura]

mm.show(amostras, titles=classes_textura, cols=3, figsize=(9, 3))

<Figure size 1350x450 with 3 Axes>

**Figure 7.18:** Échantillons synthétiques des trois classes de texture utilisées dans l


### 7.13.1 Pipeline d’extraction de caractéristiques et évaluation comparative

Pour comparer les performances des différentes formes de représentation des images, un ensemble de données équilibré contenant 60 échantillons par classe a été généré. À partir de cet ensemble, trois types de vecteurs de caractéristiques ont été extraits, chacun représentant des aspects distincts de l’information visuelle :

1. **Pixels bruts** : vecteur obtenu par l’aplatissement (*flattening*) de la matrice d’intensités de l’image, résultant en un vecteur de $64 \times 64 = 4096$ attributs ;
2. **Histogramme LBP uniforme** : histogramme normalisé des fréquences des motifs locaux produits par l’opérateur LBP uniforme, composé de 10 attributs ;
3. **Descripteur HOG** : vecteur formé par des histogrammes de gradients orientés, qui représentent la distribution spatiale des orientations des contours, totalisant 128 attributs.

Ces descripteurs ayant des échelles et des dimensionalités distinctes, les vecteurs de caractéristiques sont standardisés à l’aide du `StandardScaler`, de sorte que chaque attribut présente une moyenne nulle et un écart-type unitaire. Cette étape évite que des attributs de plus grande amplitude n’influencent de manière disproportionnée le calcul des distances euclidiennes employé par le classificateur.

L’évaluation est réalisée à l’aide de l’algorithme $k$-NN avec $k=5$, sous le même protocole de validation croisée stratifiée en cinq partitions décrit dans la section précédente. La précision moyenne obtenue sur les cinq exécutions, voir [Figure 7.19](#fig-07-comparativo-kfolds), fournit une estimation plus stable des performances du classificateur, réduisant la dépendance à une seule division entre entraînement et test.

In [18]:
rng = np.random.default_rng(42)

# 1. Génération de la base de données
X_bruto, X_lbp, X_hog, y_textura = [], [], [], []

for classe in classes_textura:
    for _ in range(60):
        img = gerar_textura(classe, ruido=0.10)
        
        # Extraction 1 : Pixels bruts
        X_bruto.append(img.ravel())
        
        # Extraction 2 : Histogramme LBP uniforme
        lbp = local_binary_pattern(img, P=8, R=1, method="uniform")
        hist_lbp, _ = np.histogram(lbp, bins=10, range=(0, 10), density=True)
        X_lbp.append(hist_lbp)
        
        # Extraction 3 : Descripteur HOG
        feat_hog = hog(img, orientations=8, pixels_per_cell=(16, 16), cells_per_block=(1, 1))
        X_hog.append(feat_hog)
        
        y_textura.append(classe)

y_textura = np.array(y_textura)
descritores = {
    "Pixels Brutos": np.array(X_bruto),
    "LBP (Textura)": np.array(X_lbp),
    "HOG (Forma)": np.array(X_hog)
}

# 2. Évaluation statistique via validation croisée 5-fold
resultados_media = {}
resultados_desvio = {}

knn = KNeighborsClassifier(n_neighbors=5)
scaler = StandardScaler()

for nome, X_dados in descritores.items():
    X_norm = scaler.fit_transform(X_dados)
    scores = cross_val_score(knn, X_norm, y_textura, cv=5, scoring="accuracy")
    resultados_media[nome] = scores.mean()
    resultados_desvio[nome] = scores.std()
    print(f"{nome:15s} -> Précision moyenne : {scores.mean():.4f} (± {scores.std():.4f})")

# 3. Tracé du graphique comparatif formel
plt.figure(figsize=(7, 4.5))
nomes_desc = list(resultados_media.keys())
medias = list(resultados_media.values())
desvios = list(resultados_desvio.values())

bars = plt.bar(nomes_desc, medias, yerr=desvios, capsize=6, 
               color=["#6366f1", "#9333ea", "#f97316"], 
               width=0.45, edgecolor="black", alpha=0.85)
plt.ylabel("Acurácia Média (5-Fold CV)", fontsize=11)
plt.title("Análise Comparativa de Descritores para Classificação de Texturas", 
          fontsize=12, fontweight="bold")
plt.ylim(0.3, 1.1)
plt.grid(axis="y", linestyle="--", alpha=0.5)

for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, 
             h + 0.03, f"{h:.3f}", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

Pixels Brutos   -> Précision moyenne : 0.6889 (± 0.0478)
LBP (Textura)   -> Précision moyenne : 0.7611 (± 0.0648)
HOG (Forma)     -> Précision moyenne : 0.4889 (± 0.0624)


<Figure size 2100x1350 with 1 Axes>

**Figure 7.19:** Précision moyenne obtenue par validation croisée (5-fold) pour les descripteurs de Pixels Bruts, LBP et HOG appliqués à la base de textures synthétiques.


> ### 📝 Pourquoi cela fonctionne-t-il ? — LBP comme représentation des textures
>
> Le descripteur LBP représente la texture d'une image au moyen d'un histogramme normalisé qui comptabilise la fréquence des motifs locaux d'intensité. Plutôt que de stocker directement les valeurs des pixels ou leurs positions, cette représentation résume la distribution des microstructures présentes dans l'image, produisant un vecteur de caractéristiques compact.
>
> Dans cette expérience, trois types de descripteurs ont été comparés : les pixels bruts, LBP et HOG. Les vecteurs formés par les **pixels bruts** préservent toutes les intensités de l'image, mais incorporent également des variations dues au bruit et aux petits déplacements spatiaux, ce qui peut rendre difficile la comparaison entre échantillons au moyen de la distance euclidienne.
>
> Le descripteur **HOG** représente la distribution des orientations des gradients, étant approprié pour décrire les formes et les contours. Comme les images utilisées dans ce projet diffèrent principalement par les propriétés de texture, et non par la présence de contours bien définis, cette représentation tend à capturer moins d'informations discriminatives que le LBP.
>
> Quant au **LBP**, il a été développé spécifiquement pour caractériser les motifs locaux de texture. Son histogramme décrit la fréquence des microstructures présentes dans l'image, indépendamment de leur position exacte, rendant la représentation moins sensible aux petites variations spatiales et aux changements monotones d'éclairage.
>
> Bien que les histogrammes des différentes classes présentent des distributions distinctes, le bruit gaussien introduit lors de la génération des images augmente la variabilité entre les échantillons de la même classe et peut produire des zones de chevauchement dans l'espace des caractéristiques. Par conséquent, certaines textures peuvent être confondues par le classificateur. Néanmoins, lorsque les caractéristiques pertinentes pour distinguer les classes sont associées aux motifs locaux de texture, on s'attend à ce que des descripteurs conçus à cette fin, comme le LBP, produisent des représentations plus informatives que celles basées uniquement sur les intensités des pixels ou les orientations des gradients.

### 7.13.2 Diagnostic fin du classificateur : précision, rappel et F1-score

L’exactitude résume la performance du classificateur en une seule valeur, mais n’indique pas comment cette performance se répartit entre les différentes classes. Pour une analyse plus détaillée, on utilise des métriques calculées individuellement pour chaque classe.

La **précision** (*precision*) mesure la proportion d’échantillons classés comme appartenant à une classe qui lui appartiennent réellement. Le **rappel** (*recall*) mesure la proportion d’échantillons de la classe qui ont été correctement identifiés par le classificateur. Le **F1-score** correspond à la moyenne harmonique entre précision et rappel, fournissant un indicateur qui équilibre ces deux mesures.

Le rapport indique également le **support** (*support*), c’est-à-dire le nombre d’échantillons de chaque classe présents dans l’ensemble de test. Cette information est importante pour contextualiser les métriques, car les résultats obtenus sur un petit nombre d’échantillons tendent à présenter une plus grande variabilité.

La [Figure 7.20](#fig-07-metricas-avaliacao-detalhadas) présente ces métriques pour les trois classes de texture. Ensemble, elles permettent d’identifier des différences de performance qui ne sont pas évidentes à partir de la seule exactitude. Par exemple, une classe peut présenter une précision élevée et un rappel plus faible, indiquant que le classificateur commet peu de faux positifs, mais ne parvient pas à identifier une partie des échantillons qui appartiennent réellement à cette classe. Ce type d’analyse aide à comprendre les limites du modèle et à identifier d’éventuelles stratégies pour son amélioration.

In [19]:
X_lbp_data = np.array(X_lbp)
y_textura_data = np.array(y_textura)

# Effectuer une division entraînement/test pour le rapport détaillé
Xt_treino, Xt_teste, yt_treino, yt_teste = train_test_split(
    X_lbp_data, y_textura_data, test_size=0.3, random_state=42, stratify=y_textura_data
)

# Mettre à l'échelle les données
scaler = StandardScaler()
Xt_treino_scaled = scaler.fit_transform(Xt_treino)
Xt_teste_scaled = scaler.transform(Xt_teste)

# Entraîner le classificateur k-NN
knn_textura = KNeighborsClassifier(n_neighbors=5)
knn_textura.fit(Xt_treino_scaled, yt_treino)

y_pred = knn_textura.predict(Xt_teste_scaled)

report = classification_report(
    yt_teste,
    y_pred,
    target_names=classes_textura,
    output_dict=True
)

print("=== RAPPORT DE CLASSIFICATION DÉTAILLÉ ===")
print(f"{'Classe':<12} {'Precisão':>10} {'Revocação':>12} {'F1-score':>10} {'Suporte':>10}")
for classe in classes_textura:
    r = report[classe]
    print(f"{classe:<12} {r['precision']:>10.2f} {r['recall']:>12.2f} "
          f"{r['f1-score']:>10.2f} {r['support']:>10.0f}")

precision = precision_score(yt_teste, y_pred, average=None, labels=classes_textura)
recall = recall_score(yt_teste, y_pred, average=None, labels=classes_textura)
f1 = f1_score(yt_teste, y_pred, average=None, labels=classes_textura)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(classes_textura))
width = 0.25

bars1 = ax.bar(x - width, precision, width, label='Precisão', color='#6366f1', alpha=0.8)
bars2 = ax.bar(x, recall, width, label='Revocação', color='#f97316', alpha=0.8)
bars3 = ax.bar(x + width, f1, width, label='F1-Score', color='#22c55e', alpha=0.8)

ax.set_xlabel('Classe', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Métricas por Classe - Classificação de Texturas', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(classes_textura)
ax.legend(loc='upper right')
ax.set_ylim(0, 1.35)
ax.grid(axis='y', alpha=0.3)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                 f'{height:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("Interprétation des métriques :")
print("- Précision : parmi les échantillons classés comme appartenant à la classe,",
      "combien étaient corrects ?")
print("- Rappel : parmi les échantillons qui appartiennent réellement à la classe, combien",
      "ont été identifiés ?")
print("- Score F1 : moyenne harmonique entre précision et rappel.")
print("- Support : nombre d'échantillons réels de chaque classe présents dans l'ensemble de test.")
print("\nLe support ne mesure pas la performance ; il indique simplement combien d'exemples de chaque",
      "classe ont été \nutilisés dans l'évaluation.")

=== RAPPORT DE CLASSIFICATION DÉTAILLÉ ===
Classe         Precisão    Revocação   F1-score    Suporte
granular           0.67         0.78       0.72         18
listrada           0.55         0.61       0.58         18
manchada           1.00         0.72       0.84         18


<Figure size 3000x1500 with 1 Axes>

**Figure 7.20:** Métriques d


Interprétation des métriques :
- Précision : parmi les échantillons classés comme appartenant à la classe, combien étaient corrects ?
- Rappel : parmi les échantillons qui appartiennent réellement à la classe, combien ont été identifiés ?
- Score F1 : moyenne harmonique entre précision et rappel.
- Support : nombre d'échantillons réels de chaque classe présents dans l'ensemble de test.

Le support ne mesure pas la performance ; il indique simplement combien d'exemples de chaque classe ont été 
utilisés dans l'évaluation.


## 7.14 Limites des descripteurs artisanaux

Les expériences de ce chapitre montrent que les descripteurs classiques peuvent
être assez efficaces dans les tâches de classification, mais présentent également
des limites importantes :

- **Spécificité :** chaque descripteur a été développé pour représenter un
  type d'information donné, comme la couleur, la texture ou la forme. Ainsi, un
  descripteur adapté à une tâche peut ne pas être le plus approprié pour une
  autre.
- **Dépendance aux hyperparamètres :** la performance de descripteurs comme
  LBP et HOG dépend du choix de paramètres, tels que le rayon de voisinage,
  le nombre de points échantillonnés, la taille de la cellule et le nombre d'orientations,
  qui doivent être ajustés selon l'application.
- **Représentation limitée :** les descripteurs de couleur, de texture et de gradient
  capturent des propriétés de bas niveau de l'image, mais ne représentent pas
  directement des concepts sémantiques plus complexes, comme les objets ou les scènes.
- **Malédiction de la dimensionnalité :** des descripteurs trop étendus peuvent
  réduire l'efficacité des classificateurs basés sur la distance, comme le
  *k*-NN.

Ces limites motivent l'évolution des techniques étudiées dans les prochains
chapitres. Le **Chapitre 8** présente des méthodes classiques pour la détection et
la correspondance de caractéristiques dans les images, tandis que le **Chapitre 9**
introduit les **Réseaux de Neurones Convolutionnels**, capables d'apprendre
automatiquement des représentations adaptées à chaque tâche à partir des
données.

## 7.15 Résumé

Dans ce chapitre, les fondements de la reconnaissance de formes appliquée aux images ont été présentés. Les principaux concepts étudiés ont été :

- ***Pipeline* de reconnaissance de formes :** acquisition, prétraitement, extraction de descripteurs, classification et évaluation.
- **Descripteurs classiques :** descripteurs de couleur, LBP pour la texture et HOG pour la forme, utilisés pour représenter différentes caractéristiques des images.
- **Normalisation des caractéristiques :** standardisation (*Z-score*) afin d'éviter que les attributs de plus grande magnitude ne dominent le calcul des distances.
- **Classifieur *k*-NN :** classification basée sur les $k$ plus proches voisins dans l'espace des caractéristiques.
- **Choix du paramètre $k$ :** influence de la valeur de $k$ sur les performances du classifieur et utilisation de la validation croisée pour sa sélection.
- **Évaluation des classifieurs :** exactitude, matrice de confusion, précision, rappel et F1-score comme métriques complémentaires de performance.
- **Limites des descripteurs artisanaux :** spécificité, dépendance aux hyperparamètres et difficulté à représenter des informations de haut niveau.

Les concepts ont été illustrés au moyen d'expériences menées sur la base publique `load_digits`, des textures synthétiques générées à des fins pédagogiques et des données simulées de descripteurs de fruits.

## 7.16 🤖 Utilisation de Gemini Notebook comme tuteur

Dans cette édition, **Gemini Notebook** est présenté comme un outil
d'aide à l'étude. Le système utilise exclusivement les documents mis à
disposition par l'auteur comme source de connaissances, permettant
d'explorer les concepts du chapitre au moyen de questions, de résumés et
d'explications liés au matériel étudié.

> ### ❗ 🎓 Étudiez avec le tuteur intelligent
>
> [🚀 ACCÉDER À GEMINI NOTEBOOK : CHAPITRE 07](https://notebooklm.google.com/notebook/494d06c6-ca01-4cd2-aa00-dbe30469308c)
>
> #### 🌐 Langue et langage de programmation
>
> Le projet de ce chapitre dans Gemini Notebook a été construit uniquement
> avec le texte en **portugais** et les exemples de code en **Python**. Si
> vous étudiez à partir de l'édition en anglais ou en français, ou si vous
> suivez le parcours en C++, les réponses du tuteur peuvent ne pas
> correspondre exactement à la version que vous lisez.
>
> #### ⚠️ Avis concernant le contenu généré par IA
>
> Les réponses fournies par Gemini Notebook peuvent contenir des
> imprécisions ou des omissions. Si nécessaire, confirmez les informations
> en utilisant le matériel de ce chapitre et d'autres sources académiques
> fiables. L'exécution des exemples pratiques présentés tout au long du
> texte reste le meilleur moyen de consolider les concepts étudiés.

## 7.17 Lista de Exercícios

Os exercícios a seguir exploram e estendem os conceitos apresentados neste capítulo por meio de adaptações dos algoritmos implementados, análises experimentais e comparações entre diferentes abordagens.

1. **(10%)** Implemente um descritor de cor (histograma RGB ou HSV, com pelo menos 16 *bins* por canal) para as três classes de frutas simuladas na [Figure 7.1](#fig-07-frutas-motivacao). Treine um classificador *k*-NN com esse descritor, compare sua acurácia com a obtida pelos descritores LBP e HOG ([Figure 7.9](#fig-07-comparacao-descritores-detalhada)) e discuta em quais situações a informação de cor é mais discriminativa.

2. **(15%)** Investigue o efeito da normalização de características (*Z-score*) sobre o desempenho do *k*-NN em um espaço de atributos heterogêneo, combinando descritores de cor, LBP e HOG em um único vetor. Compare os resultados obtidos com e sem normalização para pelo menos três valores de $k$.

3. **(15%)** Reproduza a análise de *overfitting* e *underfitting* da [Figure 7.17](#fig-07-overfitting-analysis) variando o tamanho do conjunto de treinamento (por exemplo, 20%, 50% e 80% da base `load_digits`). Discuta como a quantidade de exemplos influencia a escolha do valor de $k$.

4. **(15%)** Estenda o Projeto Prático 2 adicionando uma quarta classe sintética de textura. Avalie precisão, revocação e F1-score para cada classe, seguindo o padrão da [Figure 7.20](#fig-07-metricas-avaliacao-detalhadas), e analise o impacto da nova classe na matriz de confusão.

5. **(15%)** Implemente manualmente o classificador *k*-NN, sem utilizar `sklearn`:

   `sklearn.neighbors.KNeighborsClassifier`,

   completando a função `knn_passo_a_passo` apresentada no capítulo. Compare a acurácia e o tempo de execução da implementação manual com a implementação do `scikit-learn` em conjuntos de dados de tamanhos crescentes e relacione os resultados à maldição da dimensionalidade.

6. **(15%)** Investigue a influência dos parâmetros `orientations`, `pixels_per_cell` e `cells_per_block` do descritor HOG na base `load_digits`. Avalie pelo menos quatro combinações de parâmetros e discuta o compromisso entre dimensionalidade do descritor e desempenho do classificador.

7. **(15%)** Avalie a influência dos parâmetros $P$ (número de vizinhos) e $R$ (raio) do descritor LBP na classificação das texturas sintéticas do Projeto Prático 2, considerando $P \in \{4,8,16\}$ e $R \in \{1,2,3\}$. Analise como esses parâmetros afetam a capacidade discriminativa do descritor.

8. **(Bônus – 10%)** Implemente manualmente a validação cruzada *k-fold* para o classificador *k*-NN na base `load_digits`, sem utilizar `cross_val_score`, e compare os resultados com os obtidos pela implementação do `scikit-learn` apresentada na [Figure 7.16](#fig-07-elbow-k)..

## Références du chapitre

La fondation théorique et les expériences présentées dans ce chapitre
sont basées sur les références suivantes :

- Gonzalez (2018), pour les fondements des descripteurs statistiques de texture et des opérations de prétraitement appliquées à l'extraction de caractéristiques.
- Szeliski (2022), pour la présentation du pipeline classique de reconnaissance de formes, de l'extraction de descripteurs et de l'évaluation de classificateurs en Vision par Ordinateur.
- Duda (2001), pour les fondements théoriques de la reconnaissance de formes, du classificateur *k*-NN et de la relation entre biais et variance.
- Cover (1967), pour la formulation originale de l'algorithme des *k* plus proches voisins.
- Ojala (2002), pour la formulation du descripteur *Local Binary Patterns* (LBP) et de sa variante uniforme, utilisée dans ce chapitre.
- Dalal (2005), pour la formulation du descripteur *Histogram of Oriented Gradients* (HOG), employé dans la représentation de la forme et du contour.
- Pedregosa (2011), pour l'implémentation du classificateur *k*-NN, des métriques d'évaluation et de la validation croisée dans la bibliothèque `scikit-learn`.
- Quilici-gonzalez (2014), pour la présentation didactique de classificateurs traditionnels de Reconnaissance de Formes, tels que les Arbres de Décision, les Règles de Classification et les Machines à Vecteurs de Support (SVM), complémentaires au classificateur k-NN exploré dans ce chapitre.
- Quilici-gonzalez (2026), pour la mise à jour et l'élargissement de ces contenus dans sa 2e édition, actuellement en production.

## Références du Chapitre


COVER, T.; HART, P. **Nearest neighbor pattern classification**. 1967.

DALAL, N.; TRIGGS, B. **Histograms of oriented gradients for human detection**. 2005.

DUDA, Richard O.; HART, Peter E.; STORK, David G. **Pattern Classification**. New York, Wiley-Interscience, 2001.

GONZALEZ, R. C.; WOODS, R. E. **Digital Image Processing**. New York, Pearson, 2018.

OJALA, Timo; PIETIK\"{A}INEN, Matti; M\"{A}ENP\"{A}\"{A}, Topi. **Multiresolution Gray-Scale and Rotation Invariant Texture Classification with Local Binary Patterns**. USA, IEEE Computer Society, 2002.

PEDREGOSA, Fabian *et al*. **Scikit-learn: Machine Learning in Python**. 2011.

QUILICI-GONZALEZ, José Artur; ZAMPIROLLI, Francisco de Assis. **Sistemas Inteligentes e Mineração de Dados**. Editora UFABC, 2014.

QUILICI-GONZALEZ, José Artur; ZAMPIROLLI, Francisco de Assis; SOUZA, Fábio Rezende de. **Sistemas Inteligentes e Mineração de Dados: Do Weka ao Python**. 2026.

SZELISKI, Richard. **Computer Vision: Algorithms and Applications**. Springer, 2022.

{GOOGLE}. **{NotebookLM}**. 2025.